In [ ]:
# @title
# ============================================================================
# SHIPPING FAQ ASSISTANT – FINE‑TUNE + GRADIO WEB APP (SINGLE CELL)
# ============================================================================
# This cell will:
# 1. Install dependencies
# 2. Fine‑tune a model on 50+ shipping FAQs (if not already done)
# 3. Launch a Gradio web app with a public link
# ============================================================================



import sys, subprocess, importlib, os, pickle, shutil

# ----------------------------------------------------------------------------
# 1. Install dependencies
# ----------------------------------------------------------------------------
def install(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for p in ["sentence-transformers", "scikit-learn", "torch", "gradio"]:
    install(p)

import numpy as np
import gradio as gr
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from sklearn.metrics.pairwise import cosine_similarity

# ----------------------------------------------------------------------------
# 2. Shipping FAQ Dataset (50+ FAQs)
# ----------------------------------------------------------------------------
SHIPPING_FAQS = [
    # Tracking
    {"question": "How can I track my shipment?", "answer": "Use the tracking number from your confirmation email on our tracking page or mobile app."},
    {"question": "Where is my tracking number?", "answer": "Your tracking number is in the shipment confirmation email. Check spam if you don't see it."},
    {"question": "Tracking says 'delivered' but I didn't receive it.", "answer": "Wait 24 hours; sometimes carriers mark early. If still missing, file a claim."},
    {"question": "Can I track multiple packages at once?", "answer": "Yes, enter tracking numbers separated by commas on our tracking page."},
    {"question": "Why hasn't my tracking updated?", "answer": "Tracking may not update until the package reaches a sorting facility. Allow 24–48 hours."},
    # Rates & Costs
    {"question": "How much does shipping cost?", "answer": "Cost depends on weight, dimensions, and destination. Use our rate calculator or request a quote."},
    {"question": "Do you offer free shipping?", "answer": "Free standard shipping on orders over $50 within the continental US."},
    {"question": "What are your international shipping rates?", "answer": "International rates vary by country. Use the calculator at checkout for exact pricing."},
    {"question": "Are there additional customs fees?", "answer": "International shipments may incur customs duties/taxes, which are the recipient's responsibility."},
    {"question": "Do you offer flat‑rate shipping?", "answer": "Yes, we offer flat‑rate boxes for certain sizes. See our shipping supplies page."},
    # Delivery Times
    {"question": "How long does standard shipping take?", "answer": "3–5 business days within the continental US. International: 7–14 business days."},
    {"question": "What is expedited shipping?", "answer": "Overnight and 2‑day options available at checkout for an additional fee."},
    {"question": "Do you deliver on weekends?", "answer": "Saturday delivery is available for an extra charge with select carriers."},
    {"question": "When will my order arrive?", "answer": "Check your tracking number for an estimated delivery date."},
    {"question": "Why is my package delayed?", "answer": "Delays may occur due to weather, customs, or high volume. Tracking will show updates."},
    # Returns & Refunds
    {"question": "What is your return policy?", "answer": "Return unused items within 30 days for a full refund. Customer pays return shipping unless defective."},
    {"question": "How do I start a return?", "answer": "Visit our returns portal, enter your order number, and print a prepaid label (fee deducted)."},
    {"question": "When will I get my refund?", "answer": "Refunds are processed within 5–7 business days after we receive the returned item."},
    {"question": "Can I exchange an item?", "answer": "We don't offer direct exchanges. Return the original and place a new order."},
    {"question": "Do you refund shipping charges?", "answer": "Original shipping charges are non‑refundable unless the return is due to our error."},
    # International
    {"question": "Do you ship to Canada?", "answer": "Yes, we ship to Canada. Duties and taxes may apply."},
    {"question": "What countries do you ship to?", "answer": "We ship to over 100 countries. See the full list at checkout."},
    {"question": "How long does international shipping take?", "answer": "Typically 7–14 business days, depending on customs clearance."},
    {"question": "Who pays customs fees?", "answer": "The recipient is responsible for any customs duties or import taxes."},
    {"question": "Can I track international shipments?", "answer": "Yes, tracking is available for most destinations. Use the same tracking number."},
    # Claims & Issues
    {"question": "How do I file a claim for a lost package?", "answer": "Contact support with your tracking number after 7 days past the estimated delivery date."},
    {"question": "My package arrived damaged.", "answer": "Take photos and contact us within 48 hours. We'll file a claim and send a replacement or refund."},
    {"question": "What if I received the wrong item?", "answer": "Contact support immediately with your order number. We'll arrange a return and correct shipment."},
    {"question": "Can I cancel my order?", "answer": "Orders can be canceled within 1 hour of placement. After that, they enter processing."},
    {"question": "How do I change my delivery address?", "answer": "Address changes are only possible before shipment. Contact support ASAP."},
    # Account & Billing
    {"question": "How do I create an account?", "answer": "Click 'Sign Up' on our website and enter your email and a password."},
    {"question": "I forgot my password.", "answer": "Use the 'Forgot Password' link on the login page to reset it."},
    {"question": "What payment methods do you accept?", "answer": "Visa, MasterCard, Amex, PayPal, and wire transfers for businesses."},
    {"question": "Can I get an invoice for my order?", "answer": "Invoices are emailed after purchase. You can also download them from your account."},
    {"question": "How do I update my billing address?", "answer": "Log into your account, go to 'Addresses', and edit your billing address."},
    # Business / Bulk
    {"question": "Do you offer business accounts?", "answer": "Yes, corporate accounts get volume discounts and dedicated support. Apply on our website."},
    {"question": "How do I get a quote for bulk shipping?", "answer": "Contact our sales team with your volume and requirements."},
    {"question": "Can I schedule a pickup?", "answer": "Pickup services are available for business accounts. Schedule through your dashboard."},
    {"question": "Do you offer freight shipping?", "answer": "Yes, we partner with freight carriers for large/heavy items. Request a freight quote."},
    {"question": "What are your hours of operation?", "answer": "Customer support is available 24/7 via chat and email."},
    # Packaging & Restrictions
    {"question": "What items are prohibited?", "answer": "Hazardous materials, perishables, and illegal items. See our prohibited items list."},
    {"question": "Do you provide packaging materials?", "answer": "We sell boxes, tape, and envelopes on our supplies page."},
    {"question": "Can I ship liquids?", "answer": "Non‑hazardous liquids are allowed if properly sealed and packaged."},
    {"question": "What are the size and weight limits?", "answer": "Max weight 150 lbs per package. Max length 108 inches."},
    {"question": "How should I pack fragile items?", "answer": "Use bubble wrap, packing peanuts, and a sturdy box marked 'Fragile'."},
    # Additional
    {"question": "Do you offer insurance?", "answer": "All shipments include $100 coverage. Additional insurance can be purchased."},
    {"question": "How do I contact customer support?", "answer": "Live chat on our website, email support@shippingco.com, or call 1‑800‑555‑0199."},
    {"question": "Where can I find your terms and conditions?", "answer": "Visit the 'Legal' section at the bottom of our website."},
    {"question": "Do you have a mobile app?", "answer": "Yes, download our app from the App Store or Google Play for tracking and shipping."},
    {"question": "Can I ship to a PO Box?", "answer": "Yes, but only via USPS. Select USPS at checkout."},
]

# ----------------------------------------------------------------------------
# 3. FAQMatcher Class (with fine‑tuning)
# ----------------------------------------------------------------------------
class FAQMatcher:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        print(f"🚀 Loading model: {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.faqs = []
        self.embeddings = None
        self.model_name = model_name

    def add_faqs(self, faq_list):
        self.faqs = faq_list
        if faq_list:
            questions = [faq['question'] for faq in faq_list]
            self.embeddings = self.model.encode(questions, convert_to_numpy=True)
            print(f"✅ Encoded {len(faq_list)} FAQs. Shape: {self.embeddings.shape}")

    def match(self, question, top_k=1, min_confidence=0.5):
        if self.embeddings is None: return []
        user_emb = self.model.encode([question], convert_to_numpy=True)
        sims = cosine_similarity(user_emb, self.embeddings)[0]
        idxs = np.argsort(sims)[::-1][:top_k]
        return [{'faq_id': i+1, 'question': self.faqs[i]['question'],
                 'answer': self.faqs[i]['answer'], 'confidence': float(sims[i])}
                for i in idxs if sims[i] >= min_confidence]

    def fine_tune(self, training_pairs, epochs=8, batch_size=16):
        if not training_pairs:
            print("⚠️ No training pairs.")
            return
        loader = DataLoader(training_pairs, shuffle=True, batch_size=batch_size)
        loss = losses.CosineSimilarityLoss(self.model)
        print(f"🏋️ Fine‑tuning on {len(training_pairs)} pairs for {epochs} epochs...")
        self.model.fit(train_objectives=[(loader, loss)], epochs=epochs,
                       warmup_steps=int(len(training_pairs)*0.1), show_progress_bar=True)
        print("✅ Fine‑tuning complete!")

    def save_model(self, path='shipping_faq_model_finetuned'):
        self.model.save(path)
        print(f"💾 Model saved to {path}")

    @staticmethod
    def create_training_pairs(faqs, variations_dict):
        pairs = []
        for idx, faq in enumerate(faqs):
            if idx in variations_dict:
                orig = faq['question']
                for var in variations_dict[idx]:
                    pairs.append(InputExample(texts=[orig, var], label=1.0))
        return pairs

# ----------------------------------------------------------------------------
# 4. FAQ Variations (paraphrases for training)
# ----------------------------------------------------------------------------
FAQ_VARIATIONS = {
    0: ["track my order", "where is my package", "tracking status", "follow my shipment"],
    1: ["tracking number location", "didn't get tracking", "find my tracking code"],
    2: ["tracking says delivered but not received", "package marked delivered missing"],
    3: ["track multiple orders", "multiple tracking numbers"],
    4: ["tracking not updating", "no tracking movement"],
    5: ["shipping cost", "how much to ship", "shipping price"],
    6: ["free delivery", "no shipping charge", "complimentary shipping"],
    7: ["international shipping price", "cost to ship overseas"],
    8: ["customs fees", "import duties", "tariffs"],
    9: ["flat rate boxes", "fixed shipping price"],
    10: ["standard delivery time", "how many days for shipping"],
    11: ["express shipping", "fast delivery options", "overnight shipping"],
    12: ["saturday delivery", "weekend shipping"],
    13: ["estimated arrival", "delivery date"],
    14: ["package delayed", "why is my shipment late"],
    15: ["return items", "refund process", "send back order"],
    16: ["start a return", "how to return"],
    17: ["refund timeline", "when will I be refunded"],
    18: ["exchange product", "swap item"],
    19: ["refund shipping cost", "get shipping fee back"],
    20: ["ship to canada", "canada delivery"],
    21: ["countries served", "international destinations"],
    22: ["international delivery time", "overseas shipping duration"],
    23: ["customs responsibility", "who pays import tax"],
    24: ["track international package", "global tracking"],
    25: ["lost package claim", "file a claim for missing shipment"],
    26: ["damaged item", "broken upon arrival"],
    27: ["wrong item received", "incorrect order"],
    28: ["cancel order", "stop shipment"],
    29: ["change address", "update delivery location"],
    30: ["create account", "register"],
    31: ["password reset", "forgot login"],
    32: ["accepted payment", "credit cards"],
    33: ["get invoice", "receipt copy"],
    34: ["update billing info", "change payment address"],
    35: ["corporate account", "business shipping"],
    36: ["bulk quote", "volume discount"],
    37: ["schedule pickup", "carrier collection"],
    38: ["freight service", "large shipment"],
    39: ["support hours", "customer service availability"],
    40: ["prohibited items", "cannot ship"],
    41: ["packaging supplies", "buy boxes"],
    42: ["shipping liquids", "send liquid items"],
    43: ["weight limit", "size restrictions"],
    44: ["pack fragile", "protect breakable"],
    45: ["shipping insurance", "coverage value"],
    46: ["contact support", "help desk"],
    47: ["terms of service", "legal information"],
    48: ["mobile app", "download app"],
    49: ["ship to PO box", "PO Box delivery"],
}

# ----------------------------------------------------------------------------
# 5. Load or Fine‑Tune Model
# ----------------------------------------------------------------------------
MODEL_PATH = 'shipping_faq_model_finetuned'
matcher = FAQMatcher()

if os.path.exists(MODEL_PATH):
    print(f"📂 Loading existing fine‑tuned model from {MODEL_PATH}")
    matcher.model = SentenceTransformer(MODEL_PATH)
    matcher.add_faqs(SHIPPING_FAQS)
else:
    print("🆕 No fine‑tuned model found. Starting fine‑tuning...")
    matcher.add_faqs(SHIPPING_FAQS)
    training_pairs = FAQMatcher.create_training_pairs(SHIPPING_FAQS, FAQ_VARIATIONS)
    print(f"📚 Created {len(training_pairs)} training pairs.")
    matcher.fine_tune(training_pairs, epochs=8, batch_size=16)
    matcher.add_faqs(SHIPPING_FAQS)  # re‑encode with fine‑tuned model
    matcher.save_model(MODEL_PATH)

# ----------------------------------------------------------------------------
# 6. Gradio Web App
# ----------------------------------------------------------------------------
def answer_question(user_query, confidence_threshold):
    if not user_query or not user_query.strip():
        return "Please enter a question.", ""
    matches = matcher.match(user_query.strip(), top_k=1, min_confidence=confidence_threshold)
    if not matches:
        return (f"No high‑confidence answer found (threshold: {confidence_threshold:.0%}). "
                "Please rephrase or contact support."), ""
    best = matches[0]
    response = f"**Answer:** {best['answer']}\n\n*Confidence: {best['confidence']:.1%}*"
    faq_info = f"Matched FAQ #{best['faq_id']}: {best['question']}"
    return response, faq_info

with gr.Blocks(title="🚢 Shipping FAQ Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🚢 Shipping FAQ Assistant
    Ask any question about shipping, tracking, returns, or international deliveries.
    """)
    with gr.Row():
        with gr.Column(scale=3):
            question = gr.Textbox(label="Your Question", placeholder="e.g., 'How do I track my package?'", lines=2)
            confidence = gr.Slider(minimum=0.0, maximum=1.0, value=0.6, step=0.05,
                                   label="Confidence Threshold (higher = stricter matching)")
            submit_btn = gr.Button("🔍 Get Answer", variant="primary")
        with gr.Column(scale=4):
            answer_output = gr.Markdown(label="Answer")
            faq_info_output = gr.Textbox(label="Matched FAQ", interactive=False)
    submit_btn.click(fn=answer_question, inputs=[question, confidence], outputs=[answer_output, faq_info_output])
    gr.Examples(examples=[["Where is my package?", 0.6], ["How much does shipping cost?", 0.6],
                          ["Do you ship to Canada?", 0.6], ["I want to return an item", 0.6]],
                inputs=[question, confidence])

print("\n" + "="*70)
print("🚀 Launching Gradio app...")
print("="*70)
demo.launch(share=True, debug=False, quiet=True)

🚀 Loading model: all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📂 Loading existing fine‑tuned model from shipping_faq_model_finetuned


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Encoded 50 FAQs. Shape: (50, 384)


/tmp/ipykernel_4083/1746920076.py:243: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="🚢 Shipping FAQ Assistant", theme=gr.themes.Soft()) as demo:



🚀 Launching Gradio app...
* Running on public URL: https://391cb971d7e1c4281e.gradio.live


In [1]:
# ============================================================================
# SHIPPING FAQ ASSISTANT – FINE‑TUNE + SIMPLE GRADIO APP (NO SLIDER, NO EXAMPLES)
# ============================================================================
# This cell will:
# 1. Install dependencies
# 2. Fine‑tune a model on 50+ shipping FAQs (if not already done)
# 3. Launch a minimal Gradio web app with a public link
# ============================================================================

import sys, subprocess, importlib, os, pickle, shutil

# ----------------------------------------------------------------------------
# 1. Install dependencies
# ----------------------------------------------------------------------------
def install(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for p in ["sentence-transformers", "scikit-learn", "torch", "gradio"]:
    install(p)

import numpy as np
import gradio as gr
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from sklearn.metrics.pairwise import cosine_similarity

# ----------------------------------------------------------------------------
# 2. Shipping FAQ Dataset (50+ FAQs)
# ----------------------------------------------------------------------------
SHIPPING_FAQS = [
    # Tracking
    {"question": "How can I track my shipment?", "answer": "Use the tracking number from your confirmation email on our tracking page or mobile app."},
    {"question": "Where is my tracking number?", "answer": "Your tracking number is in the shipment confirmation email. Check spam if you don't see it."},
    {"question": "Tracking says 'delivered' but I didn't receive it.", "answer": "Wait 24 hours; sometimes carriers mark early. If still missing, file a claim."},
    {"question": "Can I track multiple packages at once?", "answer": "Yes, enter tracking numbers separated by commas on our tracking page."},
    {"question": "Why hasn't my tracking updated?", "answer": "Tracking may not update until the package reaches a sorting facility. Allow 24–48 hours."},
    # Rates & Costs
    {"question": "How much does shipping cost?", "answer": "Cost depends on weight, dimensions, and destination. Use our rate calculator or request a quote."},
    {"question": "Do you offer free shipping?", "answer": "Free standard shipping on orders over $50 within the continental US."},
    {"question": "What are your international shipping rates?", "answer": "International rates vary by country. Use the calculator at checkout for exact pricing."},
    {"question": "Are there additional customs fees?", "answer": "International shipments may incur customs duties/taxes, which are the recipient's responsibility."},
    {"question": "Do you offer flat‑rate shipping?", "answer": "Yes, we offer flat‑rate boxes for certain sizes. See our shipping supplies page."},
    # Delivery Times
    {"question": "How long does standard shipping take?", "answer": "3–5 business days within the continental US. International: 7–14 business days."},
    {"question": "What is expedited shipping?", "answer": "Overnight and 2‑day options available at checkout for an additional fee."},
    {"question": "Do you deliver on weekends?", "answer": "Saturday delivery is available for an extra charge with select carriers."},
    {"question": "When will my order arrive?", "answer": "Check your tracking number for an estimated delivery date."},
    {"question": "Why is my package delayed?", "answer": "Delays may occur due to weather, customs, or high volume. Tracking will show updates."},
    # Returns & Refunds
    {"question": "What is your return policy?", "answer": "Return unused items within 30 days for a full refund. Customer pays return shipping unless defective."},
    {"question": "How do I start a return?", "answer": "Visit our returns portal, enter your order number, and print a prepaid label (fee deducted)."},
    {"question": "When will I get my refund?", "answer": "Refunds are processed within 5–7 business days after we receive the returned item."},
    {"question": "Can I exchange an item?", "answer": "We don't offer direct exchanges. Return the original and place a new order."},
    {"question": "Do you refund shipping charges?", "answer": "Original shipping charges are non‑refundable unless the return is due to our error."},
    # International
    {"question": "Do you ship to Canada?", "answer": "Yes, we ship to Canada. Duties and taxes may apply."},
    {"question": "What countries do you ship to?", "answer": "We ship to over 100 countries. See the full list at checkout."},
    {"question": "How long does international shipping take?", "answer": "Typically 7–14 business days, depending on customs clearance."},
    {"question": "Who pays customs fees?", "answer": "The recipient is responsible for any customs duties or import taxes."},
    {"question": "Can I track international shipments?", "answer": "Yes, tracking is available for most destinations. Use the same tracking number."},
    # Claims & Issues
    {"question": "How do I file a claim for a lost package?", "answer": "Contact support with your tracking number after 7 days past the estimated delivery date."},
    {"question": "My package arrived damaged.", "answer": "Take photos and contact us within 48 hours. We'll file a claim and send a replacement or refund."},
    {"question": "What if I received the wrong item?", "answer": "Contact support immediately with your order number. We'll arrange a return and correct shipment."},
    {"question": "Can I cancel my order?", "answer": "Orders can be canceled within 1 hour of placement. After that, they enter processing."},
    {"question": "How do I change my delivery address?", "answer": "Address changes are only possible before shipment. Contact support ASAP."},
    # Account & Billing
    {"question": "How do I create an account?", "answer": "Click 'Sign Up' on our website and enter your email and a password."},
    {"question": "I forgot my password.", "answer": "Use the 'Forgot Password' link on the login page to reset it."},
    {"question": "What payment methods do you accept?", "answer": "Visa, MasterCard, Amex, PayPal, and wire transfers for businesses."},
    {"question": "Can I get an invoice for my order?", "answer": "Invoices are emailed after purchase. You can also download them from your account."},
    {"question": "How do I update my billing address?", "answer": "Log into your account, go to 'Addresses', and edit your billing address."},
    # Business / Bulk
    {"question": "Do you offer business accounts?", "answer": "Yes, corporate accounts get volume discounts and dedicated support. Apply on our website."},
    {"question": "How do I get a quote for bulk shipping?", "answer": "Contact our sales team with your volume and requirements."},
    {"question": "Can I schedule a pickup?", "answer": "Pickup services are available for business accounts. Schedule through your dashboard."},
    {"question": "Do you offer freight shipping?", "answer": "Yes, we partner with freight carriers for large/heavy items. Request a freight quote."},
    {"question": "What are your hours of operation?", "answer": "Customer support is available 24/7 via chat and email."},
    # Packaging & Restrictions
    {"question": "What items are prohibited?", "answer": "Hazardous materials, perishables, and illegal items. See our prohibited items list."},
    {"question": "Do you provide packaging materials?", "answer": "We sell boxes, tape, and envelopes on our supplies page."},
    {"question": "Can I ship liquids?", "answer": "Non‑hazardous liquids are allowed if properly sealed and packaged."},
    {"question": "What are the size and weight limits?", "answer": "Max weight 150 lbs per package. Max length 108 inches."},
    {"question": "How should I pack fragile items?", "answer": "Use bubble wrap, packing peanuts, and a sturdy box marked 'Fragile'."},
    # Additional
    {"question": "Do you offer insurance?", "answer": "All shipments include $100 coverage. Additional insurance can be purchased."},
    {"question": "How do I contact customer support?", "answer": "Live chat on our website, email support@shippingco.com, or call 1‑800‑555‑0199."},
    {"question": "Where can I find your terms and conditions?", "answer": "Visit the 'Legal' section at the bottom of our website."},
    {"question": "Do you have a mobile app?", "answer": "Yes, download our app from the App Store or Google Play for tracking and shipping."},
    {"question": "Can I ship to a PO Box?", "answer": "Yes, but only via USPS. Select USPS at checkout."},
]

# ----------------------------------------------------------------------------
# 3. FAQMatcher Class (with fine‑tuning)
# ----------------------------------------------------------------------------
class FAQMatcher:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        print(f"Loading model: {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.faqs = []
        self.embeddings = None
        self.model_name = model_name

    def add_faqs(self, faq_list):
        self.faqs = faq_list
        if faq_list:
            questions = [faq['question'] for faq in faq_list]
            self.embeddings = self.model.encode(questions, convert_to_numpy=True)
            print(f"Encoded {len(faq_list)} FAQs. Shape: {self.embeddings.shape}")

    def match(self, question, top_k=1, min_confidence=0.6):
        if self.embeddings is None:
            return []
        user_emb = self.model.encode([question], convert_to_numpy=True)
        sims = cosine_similarity(user_emb, self.embeddings)[0]
        idxs = np.argsort(sims)[::-1][:top_k]
        return [{'faq_id': i+1, 'question': self.faqs[i]['question'],
                 'answer': self.faqs[i]['answer'], 'confidence': float(sims[i])}
                for i in idxs if sims[i] >= min_confidence]

    def fine_tune(self, training_pairs, epochs=8, batch_size=16):
        if not training_pairs:
            print(" No training pairs.")
            return
        loader = DataLoader(training_pairs, shuffle=True, batch_size=batch_size)
        loss = losses.CosineSimilarityLoss(self.model)
        print(f" Fine‑tuning on {len(training_pairs)} pairs for {epochs} epochs...")
        self.model.fit(train_objectives=[(loader, loss)], epochs=epochs,
                       warmup_steps=int(len(training_pairs)*0.1), show_progress_bar=True)
        print("Fine‑tuning complete!")

    def save_model(self, path='shipping_faq_model_finetuned'):
        self.model.save(path)
        print(f" Model saved to {path}")

    @staticmethod
    def create_training_pairs(faqs, variations_dict):
        pairs = []
        for idx, faq in enumerate(faqs):
            if idx in variations_dict:
                orig = faq['question']
                for var in variations_dict[idx]:
                    pairs.append(InputExample(texts=[orig, var], label=1.0))
        return pairs

# ----------------------------------------------------------------------------
# 4. FAQ Variations (paraphrases for training)
# ----------------------------------------------------------------------------
FAQ_VARIATIONS = {
    0: ["track my order", "where is my package", "tracking status", "follow my shipment"],
    1: ["tracking number location", "didn't get tracking", "find my tracking code"],
    2: ["tracking says delivered but not received", "package marked delivered missing"],
    3: ["track multiple orders", "multiple tracking numbers"],
    4: ["tracking not updating", "no tracking movement"],
    5: ["shipping cost", "how much to ship", "shipping price"],
    6: ["free delivery", "no shipping charge", "complimentary shipping"],
    7: ["international shipping price", "cost to ship overseas"],
    8: ["customs fees", "import duties", "tariffs"],
    9: ["flat rate boxes", "fixed shipping price"],
    10: ["standard delivery time", "how many days for shipping"],
    11: ["express shipping", "fast delivery options", "overnight shipping"],
    12: ["saturday delivery", "weekend shipping"],
    13: ["estimated arrival", "delivery date"],
    14: ["package delayed", "why is my shipment late"],
    15: ["return items", "refund process", "send back order"],
    16: ["start a return", "how to return"],
    17: ["refund timeline", "when will I be refunded"],
    18: ["exchange product", "swap item"],
    19: ["refund shipping cost", "get shipping fee back"],
    20: ["ship to canada", "canada delivery"],
    21: ["countries served", "international destinations"],
    22: ["international delivery time", "overseas shipping duration"],
    23: ["customs responsibility", "who pays import tax"],
    24: ["track international package", "global tracking"],
    25: ["lost package claim", "file a claim for missing shipment"],
    26: ["damaged item", "broken upon arrival"],
    27: ["wrong item received", "incorrect order"],
    28: ["cancel order", "stop shipment"],
    29: ["change address", "update delivery location"],
    30: ["create account", "register"],
    31: ["password reset", "forgot login"],
    32: ["accepted payment", "credit cards"],
    33: ["get invoice", "receipt copy"],
    34: ["update billing info", "change payment address"],
    35: ["corporate account", "business shipping"],
    36: ["bulk quote", "volume discount"],
    37: ["schedule pickup", "carrier collection"],
    38: ["freight service", "large shipment"],
    39: ["support hours", "customer service availability"],
    40: ["prohibited items", "cannot ship"],
    41: ["packaging supplies", "buy boxes"],
    42: ["shipping liquids", "send liquid items"],
    43: ["weight limit", "size restrictions"],
    44: ["pack fragile", "protect breakable"],
    45: ["shipping insurance", "coverage value"],
    46: ["contact support", "help desk"],
    47: ["terms of service", "legal information"],
    48: ["mobile app", "download app"],
    49: ["ship to PO box", "PO Box delivery"],
}

# ----------------------------------------------------------------------------
# 5. Load or Fine‑Tune Model
# ----------------------------------------------------------------------------
MODEL_PATH = 'shipping_faq_model_finetuned'
matcher = FAQMatcher()

if os.path.exists(MODEL_PATH):
    print(f" Loading existing fine‑tuned model from {MODEL_PATH}")
    matcher.model = SentenceTransformer(MODEL_PATH)
    matcher.add_faqs(SHIPPING_FAQS)
else:
    print("No fine‑tuned model found. Starting fine‑tuning...")
    matcher.add_faqs(SHIPPING_FAQS)
    training_pairs = FAQMatcher.create_training_pairs(SHIPPING_FAQS, FAQ_VARIATIONS)
    print(f" Created {len(training_pairs)} training pairs.")
    matcher.fine_tune(training_pairs, epochs=8, batch_size=16)
    matcher.add_faqs(SHIPPING_FAQS)  # re‑encode with fine‑tuned model
    matcher.save_model(MODEL_PATH)

# ----------------------------------------------------------------------------
# 6. Simplified Gradio App
# ----------------------------------------------------------------------------
def answer_question(user_query):
    if not user_query or not user_query.strip():
        return " Please enter a question.", ""
    matches = matcher.match(user_query.strip(), top_k=1, min_confidence=0.6)
    if not matches:
        return (" No confident answer found. Please rephrase or contact support.", "")
    best = matches[0]
    response = f"**{best['answer']}**"
    info = f"Matched FAQ #{best['faq_id']}: {best['question']} (confidence: {best['confidence']:.1%})"
    return response, info

with gr.Blocks(title=" Shipping FAQ Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # Shipping FAQ Assistant
    Ask any question about shipping, tracking, returns, or international deliveries.
    """)
    with gr.Row():
        question = gr.Textbox(
            label="Your Question",
            placeholder="e.g., 'How do I track my package?'",
            lines=2,
            scale=3
        )
        submit_btn = gr.Button("🔍 Get Answer", variant="primary", scale=1)

    answer_output = gr.Markdown(label="Answer")
    faq_info_output = gr.Textbox(label="Matched FAQ", interactive=False)

    submit_btn.click(
        fn=answer_question,
        inputs=question,
        outputs=[answer_output, faq_info_output]
    )

print("\n" + "="*70)
print(" Launching Gradio app...")
print("="*70)
demo.launch(share=True, debug=False, quiet=True)

/tmp/ipykernel_757/26749042.py:26: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


Loading model: all-MiniLM-L6-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

Fine‑tuning complete!
Encoded 50 FAQs. Shape: (50, 384)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 Model saved to shipping_faq_model_finetuned


/tmp/ipykernel_757/26749042.py:238: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title=" Shipping FAQ Assistant", theme=gr.themes.Soft()) as demo:



 Launching Gradio app...
* Running on public URL: https://daf91e99a1c2dac227.gradio.live
